# 06 - GNN Forward on DB-First Leaf Graph

Run the real `src/gnn` forward pass on the active DB-first leaf graph.

This notebook is no longer a toy 8D backprop sandbox. It now:
- loads `tool_leaf_nodes` and `tool_leaf_edges_next` from the active build
- creates real 1024D seed embeddings for leaf nodes
- runs the real `gnnForward()` with `DEFAULT_GNN_CONFIG`
- persists `gnn_params` back into `.vault-exec/vault.kv`


In [ ]:
import {
  getActiveTrainingBuildId,
  listActiveToolLeafEdgesNext,
  listActiveToolLeafNodes,
} from "../src/training-data/rebuild.ts";
import { buildToolLeafGraphNodes, buildToolLeafSeedEmbeddings, runLeafGnnForward } from "../src/training-data/model-inputs.ts";
import { openVaultStore } from "../src/db/index.ts";
import { DEFAULT_GNN_CONFIG } from "../src/gnn/domain/types.ts";
import { getBlasStatus, initBlasAcceleration } from "../src/gnn/infrastructure/blas-ffi.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

const kv = await Deno.openKv(DB_PATH);
const buildId = await getActiveTrainingBuildId(kv);
if (!buildId) {
  throw new Error(`No active training build found in ${DB_PATH}. Run 'deno task cli init ${VAULT_PATH}' first.`);
}
const nodeRows = await listActiveToolLeafNodes(kv);
const edgeRows = await listActiveToolLeafEdgesNext(kv);
kv.close();

const blasReady = initBlasAcceleration();
const blas = getBlasStatus();

console.log(`Active build: ${buildId}`);
console.log(`Leaf nodes: ${nodeRows.length}`);
console.log(`Leaf transitions: ${edgeRows.length}`);
console.log(`DEFAULT_GNN_CONFIG embDim: ${DEFAULT_GNN_CONFIG.embDim}`);
console.log(`BLAS ready: ${blasReady}`);
console.log(`BLAS path: ${blas.path ?? 'n/a'}`);


In [ ]:
const seedEmbeddings = buildToolLeafSeedEmbeddings(nodeRows, DEFAULT_GNN_CONFIG.embDim);
const graphNodes = buildToolLeafGraphNodes(nodeRows, edgeRows, seedEmbeddings);

const store = await openVaultStore(DB_PATH);
let result;
let storedParams;

try {
  result = await runLeafGnnForward(store, nodeRows, edgeRows, DEFAULT_GNN_CONFIG);
  storedParams = await store.getGnnParams();
} finally {
  store.close();
}

console.log(`Params source: ${result.paramsSource}`);
console.log(`Graph nodes built: ${graphNodes.length}`);
console.log(`Forward embeddings: ${result.gnnEmbeddings.size}`);
console.log(`Persisted params: ${storedParams !== null}`);


In [ ]:
function cosine(left, right) {
  let dot = 0;
  let leftNorm = 0;
  let rightNorm = 0;
  for (let index = 0; index < left.length; index++) {
    dot += left[index] * right[index];
    leftNorm += left[index] * left[index];
    rightNorm += right[index] * right[index];
  }
  return dot / (Math.sqrt(leftNorm) * Math.sqrt(rightNorm) + 1e-9);
}

const outgoing = new Map();
for (const edge of edgeRows) {
  outgoing.set(edge.fromLeaf, (outgoing.get(edge.fromLeaf) ?? 0) + edge.weight);
}

const rows = nodeRows
  .map((row) => {
    const seed = result.seedEmbeddings.get(row.leafKey);
    const gnn = result.gnnEmbeddings.get(row.leafKey);
    return {
      leafKey: row.leafKey,
      level: row.level,
      totalOccurrences: row.totalOccurrences,
      outgoingWeight: outgoing.get(row.leafKey) ?? 0,
      cosineSeedToGnn: seed && gnn ? cosine(seed, gnn) : null,
    };
  })
  .sort((left, right) => right.totalOccurrences - left.totalOccurrences || left.leafKey.localeCompare(right.leafKey));

console.table(rows.slice(0, 20).map((row) => ({
  ...row,
  cosineSeedToGnn: row.cosineSeedToGnn?.toFixed(4) ?? 'n/a',
})));


In [ ]:
const cosineSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  title: "Seed vs GNN embedding cosine by leaf node",
  width: 720,
  height: 320,
  data: { values: rows.slice(0, 40) },
  mark: "bar",
  encoding: {
    x: { field: "cosineSeedToGnn", type: "quantitative", title: "Cosine(seed, gnn)" },
    y: { field: "leafKey", type: "nominal", sort: "-x", title: "Leaf" },
    color: { field: "level", type: "nominal", title: "Level" },
    tooltip: [
      { field: "leafKey" },
      { field: "totalOccurrences" },
      { field: "outgoingWeight" },
      { field: "cosineSeedToGnn", format: ".4f" },
    ],
  },
};

cosineSpec;
